# PCA — the directions of maximum variance

> Tutorial pair for [`pca.py`](pca.py).

## 1. Intuition
High-dimensional data often lives near a low-dimensional subspace. PCA finds the
orthogonal axes along which the data varies most and keeps the top few — the
best linear compression in a least-squares sense.

## 2. Concept (the slide)
- **Center** the data (subtract the mean).
- The **first principal component** is the unit direction of greatest variance;
  each next one is orthogonal to all previous and maximizes remaining variance.
- These directions are the **eigenvectors of the covariance matrix** (= right
  singular vectors of the centered data). Eigenvalues = variance captured.

## 3. Math derivation

Center: $\tilde X = X - \bar{\mathbf x}$. Covariance $C=\frac1n\tilde X^\top\tilde X$.

**Maximize variance.** For a unit direction $\mathbf w$, the projected variance is
$\operatorname{Var}(\tilde X\mathbf w)=\mathbf w^\top C\,\mathbf w$. Solve

$$\max_{\mathbf w}\ \mathbf w^\top C\mathbf w \quad\text{s.t.}\quad \mathbf w^\top\mathbf w=1.$$

Lagrangian $\mathcal L=\mathbf w^\top C\mathbf w-\lambda(\mathbf w^\top\mathbf w-1)$,
set $\nabla_{\mathbf w}=0$:

$$2C\mathbf w-2\lambda\mathbf w=0\;\Longrightarrow\;\boxed{C\mathbf w=\lambda\mathbf w}.$$

So $\mathbf w$ is an **eigenvector** of $C$ and the variance it captures is the
**eigenvalue** $\lambda=\mathbf w^\top C\mathbf w$. The top-$k$ eigenvectors (largest
$\lambda$) are the principal components.

**SVD route (preferred).** $\tilde X=U\Sigma V^\top$. Then
$C=\frac1n V\Sigma^2V^\top$, so columns of $V$ are the components and
$\sigma_i^2/n$ the variances — computed without forming $C$ (better conditioned).

**Reconstruction view.** PCA equivalently minimizes reconstruction error
$\sum_i\lVert\mathbf x_i-(\text{projection onto }k\text{-dim subspace})\rVert^2$ —
the same subspace. **Explained-variance ratio** $\lambda_i/\sum_j\lambda_j$ tells
you how many components to keep.

**Kernel PCA.** Replace inner products $\mathbf x^\top\mathbf x'$ with a kernel
$k(\mathbf x,\mathbf x')$ (e.g. RBF) and eigendecompose the centered kernel
(Gram) matrix — PCA in a nonlinear feature space, untangling curved manifolds.

## 4. NumPy implementation (eig, SVD, whitening, kernel PCA)

In [ ]:
# ===== actual implementation from pca.py =====
from __future__ import annotations

import numpy as np

SEED = 0

class PCANumPy:
    r"""
    Center X. The covariance is  C = (1/n) X^T X.  Its eigenvectors are the
    principal directions; eigenvalues are the variance along them.
    Equivalently, SVD  X = U S V^T  gives components V and variances S^2/n.
    """

    def __init__(self, n_components=2, method="svd", whiten=False):
        self.n_components, self.method, self.whiten = n_components, method, whiten

    def fit(self, X):
        X = np.asarray(X, float)
        self.mean_ = X.mean(0)
        Xc = X - self.mean_                              # centering is mandatory
        n = len(X)
        if self.method == "eig":
            C = (Xc.T @ Xc) / n                          # covariance
            vals, vecs = np.linalg.eigh(C)              # ascending
            order = np.argsort(vals)[::-1]
            vals, vecs = vals[order], vecs[:, order]
            self.components_ = vecs[:, :self.n_components].T
            self.explained_variance_ = vals[:self.n_components]
            total = vals.sum()
        else:  # SVD — more stable, no explicit covariance
            U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
            self.components_ = Vt[:self.n_components]
            self.explained_variance_ = (S[:self.n_components] ** 2) / n
            total = (S ** 2).sum() / n
        self.explained_variance_ratio_ = self.explained_variance_ / total
        return self

    def transform(self, X):
        Xc = np.asarray(X, float) - self.mean_
        Z = Xc @ self.components_.T
        if self.whiten:                                 # unit variance per comp
            Z = Z / np.sqrt(self.explained_variance_ + 1e-12)
        return Z

    def fit_transform(self, X):
        return self.fit(X).transform(X)

    def inverse_transform(self, Z):
        if self.whiten:
            Z = Z * np.sqrt(self.explained_variance_ + 1e-12)
        return Z @ self.components_ + self.mean_

class KernelPCANumPy:
    """Nonlinear PCA in feature space via the kernel trick (RBF)."""

    def __init__(self, n_components=2, gamma=1.0):
        self.n_components, self.gamma = n_components, gamma

    def _kernel(self, A, B):
        d2 = ((A[:, None, :] - B[None, :, :]) ** 2).sum(2)
        return np.exp(-self.gamma * d2)

    def fit_transform(self, X):
        X = np.asarray(X, float); self.X_fit = X
        n = len(X)
        K = self._kernel(X, X)
        one = np.ones((n, n)) / n
        Kc = K - one @ K - K @ one + one @ K @ one      # center in feature space
        vals, vecs = np.linalg.eigh(Kc)
        order = np.argsort(vals)[::-1]
        vals, vecs = vals[order], vecs[:, order]
        # eigenvectors scaled by 1/sqrt(eigenvalue) give unit-norm projections
        alphas = vecs[:, :self.n_components] / np.sqrt(vals[:self.n_components] + 1e-12)
        self.alphas_, self.Kc_ = alphas, Kc
        return Kc @ alphas

## 5. PyTorch implementation

In [ ]:
# ===== actual implementation from pca.py =====
import torch

def pca_torch(X, n_components=2):
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    Xt = torch.as_tensor(X, dtype=torch.float32, device=dev)
    Xc = Xt - Xt.mean(0, keepdim=True)
    U, S, Vt = torch.linalg.svd(Xc, full_matrices=False)
    comps = Vt[:n_components]
    Z = Xc @ comps.T
    return Z.cpu().numpy(), comps.cpu().numpy()

def demo():
    np.random.seed(SEED); torch.manual_seed(SEED)
    from sklearn.datasets import load_iris, make_circles

    X, y = load_iris(return_X_y=True)
    X = (X - X.mean(0)) / X.std(0)

    pe = PCANumPy(2, method="eig").fit(X)
    ps = PCANumPy(2, method="svd").fit(X)
    print("explained variance ratio (eig):", np.round(pe.explained_variance_ratio_, 3))
    print("explained variance ratio (svd):", np.round(ps.explained_variance_ratio_, 3))
    print("eig ≈ svd components:", np.allclose(np.abs(pe.components_), np.abs(ps.components_), atol=1e-4))

    Z, _ = pca_torch(X, 2)
    print("torch top-2 shape:", Z.shape)

    recon = ps.inverse_transform(ps.transform(X))
    print(f"reconstruction MSE (2 of 4 dims): {np.mean((recon - X) ** 2):.4f}")

    # kernel PCA untangles concentric circles that linear PCA cannot
    Xc, yc = make_circles(n_samples=300, factor=0.3, noise=0.05, random_state=SEED)
    Zk = KernelPCANumPy(2, gamma=10).fit_transform(Xc)
    sep = abs(Zk[yc == 0, 0].mean() - Zk[yc == 1, 0].mean())
    print(f"kernel-PCA class separation on 1st component: {sep:.3f}")

## 6. Run — eig vs SVD agreement, reconstruction, kernel PCA

In [ ]:
demo()

## 7. Visualization — 4D Iris projected to 2D + scree plot

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.datasets import load_iris
import pca as M

X, y = load_iris(return_X_y=True); X = (X - X.mean(0)) / X.std(0)
p = M.PCANumPy(n_components=4).fit(X); Z = p.transform(X)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(Z[:,0], Z[:,1], c=y, cmap="viridis", edgecolor="k", s=18)
ax[0].set_xlabel("PC1"); ax[0].set_ylabel("PC2"); ax[0].set_title("Iris in 2D")
ax[1].bar(range(1,5), p.explained_variance_ratio_)
ax[1].set_xlabel("component"); ax[1].set_ylabel("explained variance ratio")
ax[1].set_title("Scree plot")
plt.tight_layout(); plt.show()

## 8. Takeaways & pitfalls
- **Centering is mandatory**; **standardize** when features have different units.
- PCA is unsupervised (ignores labels) — for class-separating directions use LDA.
- It is **linear**; curved manifolds need kernel PCA, t-SNE, or UMAP.
- Keep enough components to explain ~95% of variance (read the scree plot).